# Reproducing every number in the paper

Public companion notebook for **"Prompt-Level Budget Constraints Do Not Bind: A
Budget-Controlled Ablation of a Gemma 4 Issue-Fixing Agent"** (Gemma 4 Developer Agent
Paper Track).

The notebook is fully self-contained: the per-task results table (386 records, six
arm-runs) is embedded below. It recomputes resolution rates, Wilson 95% intervals,
failure-mode distributions, per-repository breakdowns, prompt-token medians, and
same-task paired exact McNemar tests — and asserts every value against the numbers
printed in the paper. A clean run is therefore a verification contract: any deviation
from the paper raises `AssertionError`.

Statistical conventions are identical to the frozen analysis scripts (`stats.py`,
`analyze_ablation.py`) released with this paper:

* Wilson score interval, z = 1.96.
* Paired exact McNemar, two-sided conditional binomial with doubled tail.
* Failure-mode precedence: resolved → context overflow (`ContextWindowExceeded`) →
  budget truncation (`tool call budget` / `turns budget`) → session timeout →
  patch failed tests (non-empty patch, unresolved) → abandoned/no patch.
* Token metrics are trace `final_metrics` totals; medians over tasks with metrics.

Source repository (full artifact, frozen manifests, logs): the Project Links of the
writeup. Original code is Apache-2.0.

In [ ]:
import csv, io, math, statistics
from collections import Counter

CSV_TEXT = '''arm,split,instance_id,repo,resolved,agent_patch_size,error,tool_calls,total_llm_calls,duration_seconds,test_exit_code,prompt_tokens,completion_tokens,truncated_tool_call
B0,development,fastapi_11355,fastapi/fastapi,False,0,Agent exceeded tool call budget (10 calls),10,12,29.87,-1,63248,522,False
B0,development,fastapi_13537,fastapi/fastapi,False,789,Agent exceeded tool call budget (10 calls),10,12,53.71,1,109997,1200,False
B0,development,fastapi_14077,fastapi/fastapi,False,1379,,10,13,37.91,1,88933,881,False
B0,development,fastapi_14266,fastapi/fastapi,False,0,Agent exceeded tool call budget (10 calls),10,13,23.35,-1,19127,479,False
B0,development,fastapi_14349,fastapi/fastapi,False,0,Agent exceeded tool call budget (10 calls),10,12,38.95,-1,118151,787,False
B0,development,fastapi_14479,fastapi/fastapi,False,1945,,8,11,56.48,1,86446,1462,True
B0,development,fastapi_14512,fastapi/fastapi,False,0,Agent exceeded tool call budget (10 calls),10,12,36.91,-1,118713,780,False
B0,development,fastapi_14953,fastapi/fastapi,False,560,,5,7,36.37,2,57628,581,False
B0,development,fastapi_15280,fastapi/fastapi,False,0,Agent exceeded tool call budget (10 calls),10,12,31.79,-1,90272,622,False
B0,development,fastapi_15661,fastapi/fastapi,False,0,Agent exceeded tool call budget (10 calls),10,12,39.96,-1,89383,989,False
B0,development,fastapi_12942,fastapi/fastapi,False,0,Agent exceeded session timeout (1.0 min),0,3,63.49,-1,17348,1536,True
B0,development,fastapi_14099,fastapi/fastapi,False,0,Agent exceeded tool call budget (10 calls),10,12,36.25,-1,94125,800,False
B0,development,fastapi_14246,fastapi/fastapi,False,0,Agent exceeded tool call budget (10 calls),10,18,37.53,-1,29154,1031,False
B0,development,fastapi_14301,fastapi/fastapi,False,0,Agent exceeded tool call budget (10 calls),10,12,30.08,-1,84459,663,False
B0,development,fastapi_14482,fastapi/fastapi,False,0,Agent exceeded tool call budget (10 calls),10,12,24.96,-1,71068,508,False
B0,development,fastapi_14487,fastapi/fastapi,False,557,,10,13,47.71,1,116225,919,False
B0,development,fastapi_14851,fastapi/fastapi,False,0,Agent exceeded tool call budget (10 calls),10,12,31.73,-1,124920,495,False
B0,development,fastapi_15030,fastapi/fastapi,False,0,Agent exceeded tool call budget (10 calls),10,12,29.44,-1,82143,540,False
B0,development,fastapi_15800,fastapi/fastapi,False,0,Agent exceeded tool call budget (10 calls),10,12,28.14,-1,87437,527,False
B0,development,fastapi_5077,fastapi/fastapi,False,1362,,10,13,57.19,1,105484,1454,False
B0,development,requests_7427,psf/requests,False,0,Agent exceeded tool call budget (10 calls),10,12,31.64,-1,105757,697,False
B0,development,requests_7505,psf/requests,False,1049,,10,12,84.28,124,62978,633,False
B0,development,rich_3278,Textualize/rich,False,671,,10,13,44.06,1,93706,1115,False
B0,development,rich_3472,Textualize/rich,False,0,Agent exceeded tool call budget (10 calls),10,12,17.56,-1,16426,416,False
B0,development,rich_3535,Textualize/rich,False,0,,10,13,23.39,1,65235,464,False
B0,development,rich_3772,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",10,10,23.96,-1,81244,345,False
B0,development,rich_3894,Textualize/rich,True,1139,,4,6,25.95,0,35361,598,False
B0,development,rich_3905,Textualize/rich,True,627,,4,6,23.48,0,44931,426,False
B0,development,rich_4076,Textualize/rich,False,0,Agent exceeded tool call budget (10 calls),10,13,26.35,-1,22768,690,False
B0,development,rich_4079,Textualize/rich,False,0,Agent exceeded tool call budget (10 calls),10,13,40.51,-1,63312,964,False
B0,development,fastapi_13786,fastapi/fastapi,False,0,,10,13,53.14,2,139398,820,False
B0,development,fastapi_13920,fastapi/fastapi,False,0,Agent exceeded tool call budget (10 calls),10,12,35.33,-1,73368,573,False
B0,development,fastapi_14258,fastapi/fastapi,False,0,Agent exceeded tool call budget (10 calls),10,12,43.42,-1,103487,742,False
B0,development,fastapi_14306,fastapi/fastapi,False,0,Agent exceeded tool call budget (10 calls),10,12,46.17,-1,97737,783,False
B0,development,fastapi_14356,fastapi/fastapi,False,899,,10,13,48.23,2,92871,886,False
B0,development,fastapi_14448,fastapi/fastapi,False,0,Agent exceeded tool call budget (10 calls),10,12,41.69,-1,78629,826,False
B0,development,fastapi_14463,fastapi/fastapi,False,1064,Agent exceeded session timeout (1.0 min),4,9,77.37,1,89711,1647,False
B0,development,fastapi_14791,fastapi/fastapi,False,501,,3,5,37.82,2,34144,587,False
B0,development,fastapi_14794,fastapi/fastapi,False,1313,,10,13,52.69,2,140834,923,False
B0,development,fastapi_14873,fastapi/fastapi,False,0,,10,13,41.11,1,120738,594,False
B0,development,fastapi_15589,fastapi/fastapi,False,0,Agent exceeded tool call budget (10 calls),10,12,49.42,-1,117623,724,False
B0,development,fastapi_15745,fastapi/fastapi,False,0,Agent exceeded tool call budget (10 calls),10,12,63.89,-1,73507,1063,False
B0,development,fastapi_5624,fastapi/fastapi,False,0,Agent exceeded tool call budget (10 calls),10,12,51.27,-1,112316,984,False
B0,development,fastapi_9425,fastapi/fastapi,False,0,Agent exceeded tool call budget (10 calls),10,13,42.12,-1,24982,569,False
B0,development,rich_3061,Textualize/rich,False,0,Agent exceeded tool call budget (10 calls),10,11,34.84,-1,63085,805,False
B0,development,rich_3130,Textualize/rich,False,0,,10,14,52.63,1,102418,1180,False
B0,development,rich_3180,Textualize/rich,False,0,Agent exceeded tool call budget (10 calls),10,12,38.5,-1,77399,893,False
B0,development,rich_3470,Textualize/rich,False,0,Agent exceeded tool call budget (10 calls),10,12,33.85,-1,68419,709,False
B0,development,rich_3486,Textualize/rich,False,0,Agent exceeded tool call budget (10 calls),10,12,42.2,-1,113205,866,False
B0,development,rich_3938,Textualize/rich,False,0,Agent exceeded tool call budget (10 calls),10,12,33.88,-1,92481,658,False
B0,development,requests_6592,psf/requests,False,579,,3,5,76.8,124,26019,361,False
B0,development,requests_6644,psf/requests,False,0,Agent exceeded tool call budget (10 calls),10,12,26.6,-1,62535,607,False
B0,development,requests_6757,psf/requests,False,490,,5,7,78.42,124,30026,468,False
B0,development,requests_7205,psf/requests,False,867,,10,12,35.9,1,96500,786,False
B0,development,requests_7433,psf/requests,False,1656,,10,13,99.06,124,79308,1046,False
B0,development,rich_3043,Textualize/rich,False,0,,10,13,41.41,1,98799,814,False
B0,development,rich_3296,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",10,10,22.35,-1,77179,279,False
B0,development,rich_3468,Textualize/rich,False,0,Agent exceeded tool call budget (10 calls),10,12,30.97,-1,67935,789,False
B0,development,rich_3469,Textualize/rich,False,295,,10,13,31.68,1,76720,673,False
B0,development,rich_3471,Textualize/rich,False,950,,10,13,38.66,1,101932,875,False
B0,development,rich_3521,Textualize/rich,False,0,Agent exceeded tool call budget (10 calls),10,12,45.48,-1,91572,918,False
B0,development,rich_3675,Textualize/rich,False,0,Agent exceeded tool call budget (10 calls),10,12,29.69,-1,87142,623,False
B0,development,rich_3782,Textualize/rich,False,0,Agent exceeded tool call budget (10 calls),10,17,23.54,-1,24877,611,False
B0,development,rich_3942,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",10,10,23.88,-1,77219,304,False
B0,development,rich_3953,Textualize/rich,False,0,Agent exceeded tool call budget (10 calls),10,13,22.83,-1,18183,590,False
B0p,development,fastapi_11355,fastapi/fastapi,False,0,,19,21,52.85,2,122878,875,False
B0p,development,fastapi_13537,fastapi/fastapi,False,930,,9,13,102.52,1,123623,2805,True
B0p,development,fastapi_14077,fastapi/fastapi,True,2302,,13,15,45.74,0,108233,844,False
B0p,development,fastapi_14266,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",12,12,28.34,-1,103299,349,False
B0p,development,fastapi_14349,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",15,15,45.53,-1,160286,766,False
B0p,development,fastapi_14479,fastapi/fastapi,False,2387,,8,10,50.27,1,80954,1049,False
B0p,development,fastapi_14512,fastapi/fastapi,False,1779,,19,23,77.23,2,132009,1755,False
B0p,development,fastapi_14953,fastapi/fastapi,False,560,,7,9,36.11,2,77864,480,False
B0p,development,fastapi_15280,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",16,16,37.25,-1,168036,532,False
B0p,development,fastapi_15661,fastapi/fastapi,False,561,,16,18,50.43,2,163187,967,False
B0p,development,fastapi_12942,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",7,12,150.19,-1,123399,4212,True
B0p,development,fastapi_14099,fastapi/fastapi,False,0,,18,21,68.32,2,206946,1290,False
B0p,development,fastapi_14246,fastapi/fastapi,False,0,,19,23,57.48,2,71242,1062,False
B0p,development,fastapi_14301,fastapi/fastapi,True,618,,12,14,50.6,0,131244,774,False
B0p,development,fastapi_14482,fastapi/fastapi,False,1484,,18,20,74.61,2,170317,1514,False
B0p,development,fastapi_14487,fastapi/fastapi,False,1133,,11,13,47.5,1,109945,649,False
B0p,development,fastapi_14851,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",11,11,39.54,-1,109148,391,False
B0p,development,fastapi_15030,fastapi/fastapi,False,0,,19,21,46.5,2,204620,679,False
B0p,development,fastapi_15800,fastapi/fastapi,False,0,,19,21,56.83,1,183240,831,False
B0p,development,fastapi_5077,fastapi/fastapi,False,1832,,16,18,83.08,1,172489,1804,False
B0p,development,requests_7427,psf/requests,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",13,15,42.71,-1,151923,963,False
B0p,development,requests_7505,psf/requests,False,1049,,14,16,92.31,124,109979,786,False
B0p,development,rich_3278,Textualize/rich,False,656,Agent exceeded tool call budget (20 calls),20,21,96.55,1,224293,2590,False
B0p,development,rich_3472,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",19,21,36.23,-1,84652,684,False
B0p,development,rich_3535,Textualize/rich,False,465,,18,20,44.66,1,156420,984,False
B0p,development,rich_3772,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",10,10,40.82,-1,96902,903,False
B0p,development,rich_3894,Textualize/rich,True,1232,,5,7,30.18,0,46855,653,False
B0p,development,rich_3905,Textualize/rich,True,1499,,6,8,33.12,0,64321,679,False
B0p,development,rich_4076,Textualize/rich,False,842,,20,16,97.59,1,75175,1804,False
B0p,development,rich_4079,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",10,10,25.99,-1,78055,355,False
B0p,development,fastapi_13786,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",13,14,56.9,-1,151764,1322,True
B0p,development,fastapi_13920,fastapi/fastapi,False,0,,19,21,42.6,1,168355,825,False
B0p,development,fastapi_14258,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",12,44,135.86,-1,525311,3937,True
B0p,development,fastapi_14306,fastapi/fastapi,False,0,,17,19,48.78,1,190292,948,False
B0p,development,fastapi_14356,fastapi/fastapi,False,899,,19,21,75.61,2,190907,1845,False
B0p,development,fastapi_14448,fastapi/fastapi,False,2272,,13,15,53.13,1,119039,1164,False
B0p,development,fastapi_14463,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",8,56,124.39,-1,697304,3560,False
B0p,development,fastapi_14791,fastapi/fastapi,False,501,,3,5,39.06,2,34333,839,False
B0p,development,fastapi_14794,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",11,11,31.8,-1,107772,390,False
B0p,development,fastapi_14873,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",14,14,41.76,-1,148478,602,False
B0p,development,fastapi_15589,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",9,9,32.49,-1,90539,323,False
B0p,development,fastapi_15745,fastapi/fastapi,False,0,,19,20,77.44,2,131501,1562,False
B0p,development,fastapi_5624,fastapi/fastapi,False,0,,19,16,40.38,1,67922,798,False
B0p,development,fastapi_9425,fastapi/fastapi,False,0,,19,21,45.19,1,118810,760,False
B0p,development,requests_6592,psf/requests,False,579,,3,5,75.98,124,26051,362,False
B0p,development,requests_6644,psf/requests,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",17,17,29.8,-1,130426,569,False
B0p,development,requests_6757,psf/requests,False,630,,10,12,156.66,124,64518,1056,False
B0p,development,requests_7205,psf/requests,False,1884,,18,20,57.75,1,160382,1600,False
B0p,development,requests_7433,psf/requests,False,1806,,9,11,91.89,124,58390,887,False
B0p,development,rich_3043,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",14,14,29.52,-1,120146,464,False
B0p,development,rich_3061,Textualize/rich,False,1578,,12,17,114.43,1,126378,3415,True
B0p,development,rich_3130,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",12,12,34.07,-1,97846,510,False
B0p,development,rich_3180,Textualize/rich,False,2706,,16,19,95.2,1,163672,2806,True
B0p,development,rich_3296,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",9,9,23.86,-1,74537,289,False
B0p,development,rich_3468,Textualize/rich,False,1055,,14,18,66.62,1,144459,1783,True
B0p,development,rich_3469,Textualize/rich,False,303,,19,21,55.96,1,159276,1417,False
B0p,development,rich_3470,Textualize/rich,False,0,,18,22,70.34,1,154723,1942,False
B0p,development,rich_3471,Textualize/rich,False,0,,19,21,57.75,1,188187,1435,False
B0p,development,rich_3486,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",13,13,27.19,-1,128526,399,False
B0p,development,rich_3521,Textualize/rich,False,504,,19,21,60.36,1,194330,1402,False
B0p,development,rich_3675,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",14,21,87.75,-1,216159,2444,False
B0p,development,rich_3782,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",18,20,57.5,-1,132353,1139,False
B0p,development,rich_3938,Textualize/rich,False,0,,18,20,50.49,1,130829,1221,False
B0p,development,rich_3942,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",12,12,26.91,-1,104259,410,False
B0p,development,rich_3953,Textualize/rich,False,0,,20,24,45.63,1,67146,1101,False
B1A,development,fastapi_11355,fastapi/fastapi,False,0,,20,25,47.96,2,107692,910,False
B1A,development,fastapi_13537,fastapi/fastapi,False,0,,20,25,46.88,1,113161,1025,False
B1A,development,fastapi_14077,fastapi/fastapi,True,2268,,19,24,53.43,0,120606,1236,False
B1A,development,fastapi_14266,fastapi/fastapi,False,0,,20,25,64.3,2,166906,1358,False
B1A,development,fastapi_14349,fastapi/fastapi,False,0,,20,27,74.74,2,162585,1806,True
B1A,development,fastapi_14479,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",15,19,44.76,-1,128252,849,False
B1A,development,fastapi_14512,fastapi/fastapi,False,1755,,17,23,72.26,2,120836,1736,False
B1A,development,fastapi_14953,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",15,16,33.06,-1,128089,443,False
B1A,development,fastapi_15280,fastapi/fastapi,False,469,,20,26,68.68,1,179962,1555,True
B1A,development,fastapi_15661,fastapi/fastapi,False,0,Agent exceeded session timeout (3.0 min),16,19,190.8,-1,86335,905,False
B1A,development,fastapi_12942,fastapi/fastapi,False,0,,20,25,41.22,2,115271,975,False
B1A,development,fastapi_14099,fastapi/fastapi,False,0,,19,24,46.04,2,106913,1126,False
B1A,development,fastapi_14246,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",16,17,32.53,-1,138933,506,False
B1A,development,fastapi_14301,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",13,14,30.48,-1,104260,452,False
B1A,development,fastapi_14482,fastapi/fastapi,False,0,,20,26,56.56,2,127877,1393,False
B1A,development,fastapi_14487,fastapi/fastapi,False,1151,,17,24,54.07,1,106553,1066,False
B1A,development,fastapi_14851,fastapi/fastapi,True,0,,18,23,58.76,0,141069,1329,False
B1A,development,fastapi_15030,fastapi/fastapi,False,2235,,19,24,83.68,2,106632,2303,False
B1A,development,fastapi_15800,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",14,15,31.28,-1,114347,404,False
B1A,development,fastapi_5077,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",11,12,27.37,-1,98860,362,False
B1A,development,requests_7427,psf/requests,True,1038,,20,27,64.97,0,175743,1583,False
B1A,development,requests_7505,psf/requests,False,0,,20,26,102.95,124,140648,1043,False
B1A,development,rich_3278,Textualize/rich,False,0,,20,25,41.51,1,100690,1064,False
B1A,development,rich_3472,Textualize/rich,False,0,,20,25,46.99,1,122867,1048,False
B1A,development,rich_3535,Textualize/rich,False,0,,20,26,33.92,1,104904,790,False
B1A,development,rich_3772,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",13,14,26.29,-1,90972,424,False
B1A,development,rich_3894,Textualize/rich,True,1638,,12,20,86.69,0,112381,2412,True
B1A,development,rich_3905,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",14,18,40.11,-1,124600,750,False
B1A,development,rich_4076,Textualize/rich,False,1024,,19,24,61.98,1,123182,1538,False
B1A,development,rich_4079,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",13,14,26.62,-1,96677,379,False
B1A,development,fastapi_13786,fastapi/fastapi,False,1226,,20,30,87.49,2,159912,2289,False
B1A,development,fastapi_13920,fastapi/fastapi,False,5747,,19,24,39.72,2,99878,892,False
B1A,development,fastapi_14258,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",12,13,27.12,-1,100040,318,False
B1A,development,fastapi_14306,fastapi/fastapi,False,0,Agent exceeded tool call budget (20 calls),20,24,49.25,-1,135860,1318,False
B1A,development,fastapi_14356,fastapi/fastapi,False,0,,20,25,42.57,2,116506,1030,False
B1A,development,fastapi_14448,fastapi/fastapi,False,871,,19,24,59.34,1,112311,1475,False
B1A,development,fastapi_14463,fastapi/fastapi,True,1948,,16,22,69.42,0,187980,1600,False
B1A,development,fastapi_14791,fastapi/fastapi,False,483,,7,14,69.19,2,72066,1402,True
B1A,development,fastapi_14794,fastapi/fastapi,False,0,,20,25,43.38,1,116911,982,False
B1A,development,fastapi_14873,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",12,13,27.14,-1,101290,328,False
B1A,development,fastapi_15589,fastapi/fastapi,False,0,,20,25,48.84,1,138543,954,False
B1A,development,fastapi_15745,fastapi/fastapi,False,0,,19,24,62.51,2,168553,1264,False
B1A,development,fastapi_5624,fastapi/fastapi,False,0,,20,25,33.57,1,100545,730,False
B1A,development,fastapi_9425,fastapi/fastapi,False,0,,19,24,44.36,1,91087,845,False
B1A,development,requests_6592,psf/requests,False,1136,,6,12,89.07,124,38322,779,False
B1A,development,requests_6644,psf/requests,False,0,,20,28,68.1,1,153838,1913,True
B1A,development,requests_6757,psf/requests,False,451,,9,16,154.23,124,42844,1016,False
B1A,development,requests_7205,psf/requests,False,2348,,19,25,67.81,1,141549,1823,False
B1A,development,requests_7433,psf/requests,False,739,,17,27,126.23,124,110092,1888,False
B1A,development,rich_3043,Textualize/rich,True,5461,,17,22,53.26,0,105931,1302,False
B1A,development,rich_3061,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",12,13,23.98,-1,96383,322,False
B1A,development,rich_3130,Textualize/rich,False,0,,20,27,61.12,1,129483,1596,False
B1A,development,rich_3180,Textualize/rich,True,1061,,17,23,69.69,0,99476,1594,False
B1A,development,rich_3296,Textualize/rich,False,0,,20,26,53.09,1,164189,1121,False
B1A,development,rich_3468,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",17,18,28.91,-1,119518,496,False
B1A,development,rich_3469,Textualize/rich,False,0,,18,26,67.65,1,129338,1691,True
B1A,development,rich_3470,Textualize/rich,False,0,,20,25,53.23,1,167923,1190,False
B1A,development,rich_3471,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",15,16,27.75,-1,103420,447,False
B1A,development,rich_3486,Textualize/rich,False,0,,19,27,84.79,1,201251,2103,True
B1A,development,rich_3521,Textualize/rich,False,0,,20,28,79.09,1,179735,2092,True
B1A,development,rich_3675,Textualize/rich,False,0,,20,25,54.9,1,141913,1182,False
B1A,development,rich_3782,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",15,16,27.43,-1,110858,419,False
B1A,development,rich_3938,Textualize/rich,False,0,,20,27,72.27,1,150876,1862,True
B1A,development,rich_3942,Textualize/rich,False,0,,19,29,78.64,1,171372,1948,False
B1A,development,rich_3953,Textualize/rich,False,0,,19,24,39.22,1,92534,943,False
B1A2,development,fastapi_11355,fastapi/fastapi,False,0,,20,26,56.65,2,68899,1216,False
B1A2,development,fastapi_13537,fastapi/fastapi,False,0,Agent completed execution without calling submit_patch.,18,25,106.41,-1,105121,3101,True
B1A2,development,fastapi_14077,fastapi/fastapi,False,0,Agent exceeded tool call budget (20 calls),20,24,47.94,-1,110033,1185,False
B1A2,development,fastapi_14266,fastapi/fastapi,False,0,,20,27,56.81,2,101272,1262,False
B1A2,development,fastapi_14349,fastapi/fastapi,False,0,Agent exceeded tool call budget (20 calls),20,24,43.86,-1,74885,1035,False
B1A2,development,fastapi_14479,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",11,14,39.62,-1,100629,620,False
B1A2,development,fastapi_14512,fastapi/fastapi,False,1609,,20,28,66.62,2,113301,1724,False
B1A2,development,fastapi_14953,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",18,22,68.85,-1,129060,1472,False
B1A2,development,fastapi_15280,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",14,17,56.28,-1,157616,1127,False
B1A2,development,fastapi_15661,fastapi/fastapi,False,1332,,16,22,61.14,2,97712,1407,False
B1A2,development,fastapi_12942,fastapi/fastapi,False,1704,,20,27,73.4,2,81450,1769,False
B1A2,development,fastapi_14099,fastapi/fastapi,False,0,,18,24,59.13,2,81624,1352,False
B1A2,development,fastapi_14246,fastapi/fastapi,False,596,,19,25,54.9,2,70747,1240,False
B1A2,development,fastapi_14301,fastapi/fastapi,True,618,,20,26,70.15,0,105987,1469,False
B1A2,development,fastapi_14482,fastapi/fastapi,False,673,,17,23,71.71,2,83549,1602,False
B1A2,development,fastapi_14487,fastapi/fastapi,False,0,,19,25,70.61,1,133315,1177,False
B1A2,development,fastapi_14851,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",19,22,51.92,-1,111446,753,False
B1A2,development,fastapi_15030,fastapi/fastapi,False,2208,,19,25,117.3,2,135822,3116,False
B1A2,development,fastapi_15800,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",18,15,49.11,-1,88474,655,False
B1A2,development,fastapi_5077,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",12,14,44.09,-1,110503,567,False
B1A2,development,requests_7427,psf/requests,True,0,,20,27,40.75,0,69987,1076,False
B1A2,development,requests_7505,psf/requests,False,508,Agent exceeded turns budget (60 turns),14,60,204.64,124,449369,4634,False
B1A2,development,rich_3278,Textualize/rich,False,0,Agent exceeded tool call budget (20 calls),20,24,39.03,-1,72378,973,False
B1A2,development,rich_3472,Textualize/rich,False,0,,20,26,46.75,1,93683,1082,False
B1A2,development,rich_3535,Textualize/rich,False,0,,20,32,87.36,1,156569,2548,False
B1A2,development,rich_3772,Textualize/rich,False,0,Agent completed execution without calling submit_patch.,19,27,107.33,-1,115801,3106,True
B1A2,development,rich_3894,Textualize/rich,True,1568,,18,25,75.68,0,177210,1779,False
B1A2,development,rich_3905,Textualize/rich,True,1585,,15,21,58.46,0,109520,1337,False
B1A2,development,rich_4076,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",15,18,36.71,-1,108602,581,False
B1A2,development,rich_4079,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",12,15,32.94,-1,84483,488,False
B1A2,development,fastapi_13786,fastapi/fastapi,False,1875,,18,26,114.21,2,163993,3385,True
B1A2,development,fastapi_13920,fastapi/fastapi,False,0,,20,25,47.0,1,62702,1189,False
B1A2,development,fastapi_14258,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",17,20,38.85,-1,96708,639,False
B1A2,development,fastapi_14306,fastapi/fastapi,False,0,,20,27,47.54,1,100029,1118,False
B1A2,development,fastapi_14356,fastapi/fastapi,False,0,Agent exceeded tool call budget (20 calls),20,24,44.88,-1,91729,1204,False
B1A2,development,fastapi_14448,fastapi/fastapi,False,0,,20,27,50.16,1,74201,1242,False
B1A2,development,fastapi_14463,fastapi/fastapi,True,2415,,12,18,68.27,0,126841,1801,False
B1A2,development,fastapi_14791,fastapi/fastapi,False,516,,8,13,64.76,2,99374,1116,False
B1A2,development,fastapi_14794,fastapi/fastapi,False,0,,20,28,62.29,1,100093,1379,False
B1A2,development,fastapi_14873,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",19,22,41.44,-1,129265,771,False
B1A2,development,fastapi_15589,fastapi/fastapi,False,0,,20,27,64.8,1,138185,1535,False
B1A2,development,fastapi_15745,fastapi/fastapi,False,0,,13,21,56.77,2,104919,1396,False
B1A2,development,fastapi_5624,fastapi/fastapi,False,907,,18,24,56.33,1,138245,1394,False
B1A2,development,fastapi_9425,fastapi/fastapi,False,620,,19,25,58.67,1,84469,1360,False
B1A2,development,requests_6592,psf/requests,False,0,,20,27,99.45,124,68773,1167,False
B1A2,development,requests_6644,psf/requests,False,733,,19,25,47.36,1,78165,1326,False
B1A2,development,requests_6757,psf/requests,False,500,,8,14,155.26,124,41024,1030,False
B1A2,development,requests_7205,psf/requests,False,0,,20,28,46.12,1,95791,1282,False
B1A2,development,requests_7433,psf/requests,False,560,,18,24,103.38,124,68361,1313,False
B1A2,development,rich_3043,Textualize/rich,True,449,,19,26,49.73,0,94346,1303,False
B1A2,development,rich_3061,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",13,16,30.14,-1,95206,495,False
B1A2,development,rich_3130,Textualize/rich,False,728,,17,23,56.34,1,90691,1367,False
B1A2,development,rich_3180,Textualize/rich,False,0,,19,25,55.35,1,105854,1394,False
B1A2,development,rich_3296,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",17,21,40.57,-1,102106,926,False
B1A2,development,rich_3468,Textualize/rich,False,0,,20,27,64.26,1,159453,1500,False
B1A2,development,rich_3469,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",11,14,28.68,-1,83525,433,False
B1A2,development,rich_3470,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",17,20,34.26,-1,123715,572,False
B1A2,development,rich_3471,Textualize/rich,False,0,,20,27,46.52,1,103294,1122,False
B1A2,development,rich_3486,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",16,19,36.36,-1,99443,681,False
B1A2,development,rich_3521,Textualize/rich,False,1286,,20,26,62.12,2,90152,1766,False
B1A2,development,rich_3675,Textualize/rich,False,2042,,18,24,62.23,1,91897,1620,False
B1A2,development,rich_3782,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",15,18,30.6,-1,86032,543,False
B1A2,development,rich_3938,Textualize/rich,False,0,,20,27,38.62,1,65310,1010,False
B1A2,development,rich_3942,Textualize/rich,False,0,,18,24,44.56,1,99524,985,False
B1A2,development,rich_3953,Textualize/rich,False,0,,20,27,38.59,1,64198,1032,False
B0p,heldout,fastapi_11194,fastapi/fastapi,False,0,,19,21,53.89,1,171267,1127,False
B0p,heldout,fastapi_13207,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",9,9,40.28,-1,86420,699,False
B0p,heldout,fastapi_13713,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",10,10,26.19,-1,101380,372,False
B0p,heldout,fastapi_14186,fastapi/fastapi,False,713,,12,17,112.03,2,170859,3257,True
B0p,heldout,fastapi_14262,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",10,10,25.71,-1,87986,293,False
B0p,heldout,fastapi_14297,fastapi/fastapi,False,2169,,6,8,49.53,2,68034,1217,False
B0p,heldout,fastapi_14303,fastapi/fastapi,False,2003,,4,6,37.35,2,39226,897,False
B0p,heldout,fastapi_14360,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",12,12,28.21,-1,103424,414,False
B0p,heldout,fastapi_14361,fastapi/fastapi,False,512,,10,12,38.47,2,120473,599,False
B0p,heldout,fastapi_14371,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",11,11,29.24,-1,92533,417,False
B0p,heldout,fastapi_14372,fastapi/fastapi,True,1420,,15,17,48.44,0,154451,1029,False
B0p,heldout,fastapi_14419,fastapi/fastapi,False,1479,,17,20,87.96,1,200281,2273,True
B0p,heldout,fastapi_14430,fastapi/fastapi,False,765,,19,23,48.8,2,66063,1168,False
B0p,heldout,fastapi_14455,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",16,16,51.77,-1,144640,1237,False
B0p,heldout,fastapi_14458,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",10,12,93.49,-1,116303,2561,True
B0p,heldout,fastapi_14459,fastapi/fastapi,False,0,,19,23,49.41,2,75921,1153,False
B0p,heldout,fastapi_14485,fastapi/fastapi,False,0,Agent exceeded tool call budget (20 calls),20,21,32.81,-1,122154,735,False
B0p,heldout,fastapi_14492,fastapi/fastapi,True,391,,10,12,31.08,0,112882,522,False
B0p,heldout,fastapi_14583,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",9,9,24.47,-1,95069,292,False
B0p,heldout,fastapi_14605,fastapi/fastapi,False,2058,,18,20,47.42,2,195665,1012,False
B0p,heldout,fastapi_14609,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",10,10,27.3,-1,100491,284,False
B0p,heldout,fastapi_14616,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",12,12,28.33,-1,101732,414,False
B0p,heldout,fastapi_14786,fastapi/fastapi,True,404,,4,6,23.16,0,37884,345,False
B0p,heldout,fastapi_14962,fastapi/fastapi,False,0,Agent exceeded tool call budget (20 calls),20,23,54.0,-1,77348,1428,False
B0p,heldout,fastapi_14964,fastapi/fastapi,False,0,Agent exceeded turns budget (60 turns),6,60,125.28,-1,714829,3668,False
B0p,heldout,fastapi_14978,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",16,16,30.66,-1,150295,453,False
B0p,heldout,fastapi_14986,fastapi/fastapi,False,755,,13,22,95.06,1,248479,2558,False
B0p,heldout,fastapi_15023,fastapi/fastapi,False,0,,19,21,41.69,2,141890,993,False
B0p,heldout,fastapi_15588,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",10,10,26.52,-1,89767,288,False
B0p,heldout,fastapi_15763,fastapi/fastapi,False,0,Agent exceeded session timeout (3.0 min),2,48,185.66,-1,492348,5970,False
B0p,heldout,fastapi_15785,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",15,15,34.09,-1,149596,513,False
B0p,heldout,fastapi_9555,fastapi/fastapi,False,0,Agent completed execution without calling submit_patch.,7,11,78.43,-1,108375,2268,True
B0p,heldout,fastapi_9753,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",17,17,33.81,-1,164309,568,False
B0p,heldout,requests_6589,psf/requests,False,0,Agent exceeded tool call budget (20 calls),20,22,52.1,-1,166707,1415,False
B0p,heldout,requests_6629,psf/requests,False,1492,,17,19,178.99,124,147878,3897,False
B0p,heldout,requests_7309,psf/requests,True,1756,,15,17,47.82,0,102991,1344,False
B0p,heldout,requests_7315,psf/requests,True,1533,,16,18,53.44,0,182915,1386,False
B0p,heldout,requests_7328,psf/requests,False,495,,16,18,95.54,124,193767,810,False
B0p,heldout,requests_7502,psf/requests,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",13,13,25.46,-1,108048,455,False
B0p,heldout,rich_2725,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",9,9,20.98,-1,80349,224,False
B0p,heldout,rich_2943,Textualize/rich,True,400,,15,17,42.3,0,161860,832,False
B0p,heldout,rich_3006,Textualize/rich,False,1977,,18,20,48.47,1,182995,1072,False
B0p,heldout,rich_3052,Textualize/rich,False,5195,,15,21,142.52,1,214453,4524,True
B0p,heldout,rich_3063,Textualize/rich,False,2530,Agent exceeded session timeout (3.0 min),15,42,184.96,1,438706,5850,True
B0p,heldout,rich_3064,Textualize/rich,False,475,,13,15,35.47,1,114226,745,False
B0p,heldout,rich_3067,Textualize/rich,True,655,,19,21,49.66,0,217814,1079,False
B0p,heldout,rich_3105,Textualize/rich,False,0,,19,23,45.3,1,128278,907,False
B0p,heldout,rich_3454,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",5,34,118.29,-1,419905,3644,False
B0p,heldout,rich_3480,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",12,12,26.51,-1,101264,412,False
B0p,heldout,rich_3506,Textualize/rich,False,0,,18,21,64.78,1,198930,1671,False
B0p,heldout,rich_3518,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",9,9,22.72,-1,75278,294,False
B0p,heldout,rich_3676,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",11,11,25.78,-1,92942,376,False
B0p,heldout,rich_3718,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",7,56,158.17,-1,631250,5001,False
B0p,heldout,rich_3777,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",17,17,29.74,-1,157038,526,False
B0p,heldout,rich_3882,Textualize/rich,True,1576,,11,14,49.94,0,98273,1347,True
B0p,heldout,rich_3930,Textualize/rich,False,0,,19,23,38.53,2,59506,966,False
B0p,heldout,rich_3934,Textualize/rich,False,0,,19,21,53.03,1,169982,1310,False
B0p,heldout,rich_3935,Textualize/rich,False,596,,16,18,45.71,1,108516,1192,False
B0p,heldout,rich_3944,Textualize/rich,False,1049,,18,23,60.98,1,67236,1793,True
B0p,heldout,rich_4006,Textualize/rich,False,485,,19,26,123.85,124,207074,1787,False
B0p,heldout,rich_4070,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",7,7,17.65,-1,69799,169,False
B0p,heldout,rich_4075,Textualize/rich,False,0,,18,20,50.17,1,128635,1258,False
B0p,heldout,rich_4077,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",13,13,26.59,-1,114383,424,False
B1A2,heldout,fastapi_11194,fastapi/fastapi,False,966,,20,25,69.42,1,87472,1382,False
B1A2,heldout,fastapi_13207,fastapi/fastapi,False,714,,19,25,82.96,2,122692,1665,False
B1A2,heldout,fastapi_13713,fastapi/fastapi,False,0,,19,25,67.66,2,127424,1382,False
B1A2,heldout,fastapi_14186,fastapi/fastapi,False,0,,20,26,64.32,2,113154,1382,False
B1A2,heldout,fastapi_14262,fastapi/fastapi,False,0,,17,23,64.01,2,99372,1253,False
B1A2,heldout,fastapi_14297,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",12,16,75.16,-1,134758,1596,False
B1A2,heldout,fastapi_14303,fastapi/fastapi,False,1667,,6,13,58.67,2,62188,1236,False
B1A2,heldout,fastapi_14360,fastapi/fastapi,False,0,,20,26,57.98,2,84443,1221,False
B1A2,heldout,fastapi_14361,fastapi/fastapi,False,1082,,16,22,78.2,2,113618,1725,False
B1A2,heldout,fastapi_14371,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",17,22,64.0,-1,120838,1130,False
B1A2,heldout,fastapi_14372,fastapi/fastapi,False,0,Agent exceeded tool call budget (20 calls),20,21,71.34,-1,74075,1559,False
B1A2,heldout,fastapi_14419,fastapi/fastapi,False,0,,20,26,69.54,1,112307,1438,False
B1A2,heldout,fastapi_14430,fastapi/fastapi,False,765,,19,26,65.96,2,88731,1456,False
B1A2,heldout,fastapi_14455,fastapi/fastapi,False,0,,20,27,78.03,2,83547,1731,False
B1A2,heldout,fastapi_14458,fastapi/fastapi,False,0,Agent completed execution without calling submit_patch.,17,25,109.6,-1,88877,3040,True
B1A2,heldout,fastapi_14459,fastapi/fastapi,False,0,,20,26,59.17,2,68792,1291,False
B1A2,heldout,fastapi_14485,fastapi/fastapi,False,0,,19,25,64.42,2,68423,1419,False
B1A2,heldout,fastapi_14492,fastapi/fastapi,False,944,"Failed to apply test_patch: git apply /tmp/swegemma_sandbox_98806651-8a4_2vw75lh9/tmp/tmpty9bq4c5.patch failed (1):
error: patch failed: tests/test_tutorial/test_metadata/test_tutorial001_1.py:28
error: tests/test_tutorial/test_metadata/test_tutorial001_1.py: patch does not apply


git apply -3 /tmp/swegemma_sandbox_98806651-8a4_2vw75lh9/tmp/tmpty9bq4c5.patch failed (1):
error: tests/test_tutorial/test_metadata/test_tutorial001_1.py: does not match index


git apply --ignore-space-change --ignore-whitespace /tmp/swegemma_sandbox_98806651-8a4_2vw75lh9/tmp/tmpty9bq4c5.patch failed (1):
error: patch failed: tests/test_tutorial/test_metadata/test_tutorial001_1.py:28
error: tests/test_tutorial/test_metadata/test_tutorial001_1.py: patch does not apply


git apply --recount /tmp/swegemma_sandbox_98806651-8a4_2vw75lh9/tmp/tmpty9bq4c5.patch failed (1):
error: patch failed: tests/test_tutorial/test_metadata/test_tutorial001_1.py:28
error: tests/test_tutorial/test_metadata/test_tutorial001_1.py: patch does not apply


git apply -p0 /tmp/swegemma_sandbox_98806651-8a4_2vw75lh9/tmp/tmpty9bq4c5.patch failed (1):
error: b/tests/test_tutorial/test_metadata/test_tutorial001_1.py: No such file or directory


git apply -p0 -3 /tmp/swegemma_sandbox_98806651-8a4_2vw75lh9/tmp/tmpty9bq4c5.patch failed (1):
error: b/tests/test_tutorial/test_metadata/test_tutorial001_1.py: does not exist in index


git apply -p0 --ignore-space-change --ignore-whitespace /tmp/swegemma_sandbox_98806651-8a4_2vw75lh9/tmp/tmpty9bq4c5.patch failed (1):
error: b/tests/test_tutorial/test_metadata/test_tutorial001_1.py: No such file or directory


git apply -p0 --recount /tmp/swegemma_sandbox_98806651-8a4_2vw75lh9/tmp/tmpty9bq4c5.patch failed (1):
error: b/tests/test_tutorial/test_metadata/test_tutorial001_1.py: No such file or directory


patch -p1 --batch --forward -i /tmp/swegemma_sandbox_98806651-8a4_2vw75lh9/tmp/tmpty9bq4c5.patch --dry-run dry-run failed (1):

checking file tests/test_tutorial/test_metadata/test_tutorial001_1.py
Reversed (or previously applied) patch detected!  Skipping patch.
1 out of 1 hunk ignored

patch -p1 -l --batch --forward -i /tmp/swegemma_sandbox_98806651-8a4_2vw75lh9/tmp/tmpty9bq4c5.patch --dry-run dry-run failed (1):

checking file tests/test_tutorial/test_metadata/test_tutorial001_1.py
Reversed (or previously applied) patch detected!  Skipping patch.
1 out of 1 hunk ignored

patch -p0 --batch --forward -i /tmp/swegemma_sandbox_98806651-8a4_2vw75lh9/tmp/tmpty9bq4c5.patch --dry-run dry-run failed (1):

can't find file to patch at input line 3
Perhaps you used the wrong -p or --strip option?
The text leading up to this was:
--------------------------
|--- a/tests/test_tutorial/test_metadata/test_tutorial001_1.py
|+++ b/tests/test_tutorial/test_metadata/test_tutorial001_1.py
--------------------------
No file to patch.  Skipping patch.
1 out of 1 hunk ignored

patch -p0 -l --batch --forward -i /tmp/swegemma_sandbox_98806651-8a4_2vw75lh9/tmp/tmpty9bq4c5.patch --dry-run dry-run failed (1):

can't find file to patch at input line 3
Perhaps you used the wrong -p or --strip option?
The text leading up to this was:
--------------------------
|--- a/tests/test_tutorial/test_metadata/test_tutorial001_1.py
|+++ b/tests/test_tutorial/test_metadata/test_tutorial001_1.py
--------------------------
No file to patch.  Skipping patch.
1 out of 1 hunk ignored

",15,21,68.71,-1,124914,1056,False
B1A2,heldout,fastapi_14583,fastapi/fastapi,False,856,,14,20,70.03,2,76045,1447,False
B1A2,heldout,fastapi_14605,fastapi/fastapi,False,492,,19,25,61.26,2,89081,1102,False
B1A2,heldout,fastapi_14609,fastapi/fastapi,False,74784,,19,25,77.55,2,150264,1498,False
B1A2,heldout,fastapi_14616,fastapi/fastapi,False,0,Agent exceeded tool call budget (20 calls),20,24,55.43,-1,93656,1124,False
B1A2,heldout,fastapi_14786,fastapi/fastapi,True,1627,,9,14,64.22,0,75212,1312,False
B1A2,heldout,fastapi_14962,fastapi/fastapi,False,0,,14,22,61.89,2,89193,1124,False
B1A2,heldout,fastapi_14964,fastapi/fastapi,False,2630,,15,19,68.22,2,90029,1569,False
B1A2,heldout,fastapi_14978,fastapi/fastapi,False,0,,20,26,79.79,2,152481,1408,False
B1A2,heldout,fastapi_14986,fastapi/fastapi,False,786,,19,19,70.88,1,79123,1222,False
B1A2,heldout,fastapi_15023,fastapi/fastapi,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",14,17,49.51,-1,102138,491,False
B1A2,heldout,fastapi_15588,fastapi/fastapi,False,2416,,19,25,112.53,1,171774,2360,False
B1A2,heldout,fastapi_15763,fastapi/fastapi,False,495,,20,26,83.89,1,162598,1425,False
B1A2,heldout,fastapi_15785,fastapi/fastapi,False,0,,20,24,79.46,2,145269,1122,False
B1A2,heldout,fastapi_9555,fastapi/fastapi,True,1757,,20,27,100.17,0,135225,2434,True
B1A2,heldout,fastapi_9753,fastapi/fastapi,False,0,,20,27,65.46,1,73607,1198,False
B1A2,heldout,requests_6589,psf/requests,False,1277,,18,24,111.69,124,71087,1497,False
B1A2,heldout,requests_6629,psf/requests,False,711,,20,28,115.38,124,114843,1629,False
B1A2,heldout,requests_7309,psf/requests,True,692,,19,25,57.45,0,89859,1475,False
B1A2,heldout,requests_7315,psf/requests,True,522,,19,25,50.04,0,73464,1275,False
B1A2,heldout,requests_7328,psf/requests,False,0,,20,28,103.03,124,80533,1160,False
B1A2,heldout,requests_7502,psf/requests,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",19,22,35.2,-1,119963,684,False
B1A2,heldout,rich_2725,Textualize/rich,True,1099,,20,26,81.87,0,103965,1999,False
B1A2,heldout,rich_2943,Textualize/rich,True,400,,19,25,55.3,0,121754,1308,False
B1A2,heldout,rich_3006,Textualize/rich,False,0,Agent exceeded tool call budget (20 calls),20,24,44.68,-1,93065,1108,False
B1A2,heldout,rich_3052,Textualize/rich,False,3930,,15,24,155.88,1,176059,4744,True
B1A2,heldout,rich_3063,Textualize/rich,False,0,,20,29,64.69,1,123137,1556,False
B1A2,heldout,rich_3064,Textualize/rich,False,475,,20,26,49.23,1,98609,1053,False
B1A2,heldout,rich_3067,Textualize/rich,False,0,,20,27,43.53,1,64421,1041,False
B1A2,heldout,rich_3105,Textualize/rich,False,415,,18,25,78.58,1,125963,2022,True
B1A2,heldout,rich_3454,Textualize/rich,True,1155,,19,25,70.12,0,129811,1559,False
B1A2,heldout,rich_3480,Textualize/rich,False,0,,20,28,109.72,124,118918,1121,False
B1A2,heldout,rich_3506,Textualize/rich,False,0,,20,27,50.11,1,86518,1108,False
B1A2,heldout,rich_3518,Textualize/rich,True,529,,18,25,57.74,0,129778,1368,False
B1A2,heldout,rich_3676,Textualize/rich,False,0,,20,28,65.9,1,117594,1454,False
B1A2,heldout,rich_3718,Textualize/rich,True,799,,17,23,59.35,0,87630,1444,False
B1A2,heldout,rich_3777,Textualize/rich,False,519,,20,26,58.9,1,116710,1470,False
B1A2,heldout,rich_3882,Textualize/rich,True,503,,19,25,45.76,0,71250,1106,False
B1A2,heldout,rich_3930,Textualize/rich,False,0,,18,24,61.1,2,81146,1501,False
B1A2,heldout,rich_3934,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",18,21,37.37,-1,134998,551,False
B1A2,heldout,rich_3935,Textualize/rich,False,876,,20,26,65.47,1,95702,1517,False
B1A2,heldout,rich_3944,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",15,17,37.96,-1,104831,610,False
B1A2,heldout,rich_4006,Textualize/rich,False,0,Agent exceeded tool call budget (20 calls),20,24,51.68,-1,103559,1087,False
B1A2,heldout,rich_4070,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",9,14,95.21,-1,121898,2695,True
B1A2,heldout,rich_4075,Textualize/rich,False,0,"Sandbox execution error: litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 16384 tokens. However, you requested 512 output tokens and your prompt contains at least 15873 input tokens, for a total of at least 16385 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=15873) LiteLLM Retried: 5 times",19,22,38.71,-1,117715,679,False
B1A2,heldout,rich_4077,Textualize/rich,False,1516,,19,25,59.04,1,68715,1670,False
'''

rows = list(csv.DictReader(io.StringIO(CSV_TEXT)))
by = {}
for r in rows:
    by.setdefault((r["arm"], r["split"]), []).append(r)
print(f"loaded {len(rows)} per-task records across {len(by)} arm-runs")
for k in by:
    print(" ", k, len(by[k]))

In [ ]:
def wilson(k, n, z=1.96):
    p = k / n; den = 1 + z*z/n
    c = (p + z*z/(2*n)) / den
    hw = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / den
    return (max(0.0, c-hw), min(1.0, c+hw))

def classify(r):
    if r["resolved"] == "True":
        return "resolved"
    e = r["error"]
    if "ContextWindowExceeded" in e: return "context_overflow"
    if "tool call budget" in e or "turns budget" in e: return "budget_truncated"
    if "session timeout" in e: return "session_timeout"
    if int(r["agent_patch_size"]) > 0: return "patch_failed_tests"
    return "abandoned_no_patch"

def mcnemar_exact(ra, rb):
    win = sum(1 for a, b in zip(ra, rb) if a and not b)
    lose = sum(1 for a, b in zip(ra, rb) if b and not a)
    n = win + lose
    if n == 0: return win, lose, 1.0
    k = min(win, lose)
    return win, lose, min(1.0, sum(math.comb(n, i) for i in range(k+1)) * 0.5**n * 2)

def paired(a_key, b_key):
    da = {r["instance_id"]: r["resolved"] == "True" for r in by[a_key]}
    db = {r["instance_id"]: r["resolved"] == "True" for r in by[b_key]}
    common = sorted(set(da) & set(db))
    return mcnemar_exact([da[t] for t in common], [db[t] for t in common]), len(common)

def med(key, field):
    v = sorted(int(r[field]) for r in by[key] if r[field])
    return statistics.median(v) if v else None

## Development results (n=65 per arm) — paper Table 2

In [ ]:
dev = {}
for key in [("B0","development"), ("B0p","development"), ("B1A","development"), ("B1A2","development")]:
    rr = by[key]
    ok = sum(1 for r in rr if r["resolved"] == "True")
    dev[key] = ok
    modes = Counter(classify(r) for r in rr)
    lo, hi = wilson(ok, len(rr))
    print(f"{key[0]:5s} resolved {ok}/{len(rr)} = {ok/len(rr):.1%}  Wilson 95% [{lo:.1%}, {hi:.1%}]  "
          f"overflow {modes['context_overflow']}  abandoned {modes['abandoned_no_patch']}  "
          f"prompt-median {med(key, 'prompt_tokens')}")

assert (dev[("B0","development")], dev[("B0p","development")],
        dev[("B1A","development")], dev[("B1A2","development")]) == (2, 4, 7, 6)
assert med(("B0","development"), "prompt_tokens") == 82143
assert med(("B0p","development"), "prompt_tokens") == 128526
assert med(("B1A","development"), "prompt_tokens") == 116911
assert med(("B1A2","development"), "prompt_tokens") == 99443
print("development counts and token medians: OK (2/65, 4/65, 7/65, 6/65; 82143/128526/116911/99443)")

## Paired McNemar against the same-budget control (B0′)

In [ ]:
(w1, l1, p1), n1 = paired(("B0p","development"), ("B1A","development"))
(w2, l2, p2), n2 = paired(("B0p","development"), ("B1A2","development"))
(w3, l3, p3), n3 = paired(("B0p","development"), ("B0","development"))
print(f"development B1-A  vs B0p : {w1}/{l1} wins, p={p1:.4f} (paper: 2/5, 0.4531)")
print(f"development B1-A' vs B0p : {w2}/{l2} wins, p={p2:.4f} (paper: 1/3, 0.6250)")
print(f"development B0p   vs B0  : {w3}/{l3} wins, p={p3:.4f} (paper: 2/0, 0.5000)")
assert (w1, l1, round(p1, 4), n1) == (2, 5, 0.4531, 65)
assert (w2, l2, round(p2, 4), n2) == (1, 3, 0.625, 65)
assert (w3, l3, round(p3, 2), n3) == (2, 0, 0.5, 65)
print("all three decision-line comparisons reproduced: none reaches >=10pp and p<0.05")

## Frozen held-out results (n=63 per arm) — paper Table 5

In [ ]:
hok = {}
for key in [("B0p","heldout"), ("B1A2","heldout")]:
    rr = by[key]
    ok = sum(1 for r in rr if r["resolved"] == "True")
    hok[key] = ok
    lo, hi = wilson(ok, len(rr))
    print(f"{key[0]:5s} resolved {ok}/{len(rr)} = {ok/len(rr):.1%}  Wilson 95% [{lo:.1%}, {hi:.1%}]  "
          f"overflow {sum(1 for r in rr if classify(r)=='context_overflow')}  "
          f"abandoned {sum(1 for r in rr if classify(r)=='abandoned_no_patch')}  "
          f"prompt-median {med(key, 'prompt_tokens')}")
(w4, l4, p4), n4 = paired(("B0p","heldout"), ("B1A2","heldout"))
print(f"held-out B1-A' vs B0p: {w4}/{l4} wins, p={p4:.4f}, delta {(hok[('B1A2','heldout')]-hok[('B0p','heldout')])/63*100:+.1f} pp (paper: 3/5, 0.7266, +3.2 pp)")
assert (hok[("B0p","heldout")], hok[("B1A2","heldout")]) == (8, 10)
assert med(("B0p","heldout"), "prompt_tokens") == 120473
assert med(("B1A2","heldout"), "prompt_tokens") == 102138
assert (w4, l4, round(p4, 4), n4) == (3, 5, 0.7266, 63)
print("held-out verdict reproduced: pre-registered line (>=10pp and p<0.05) not met")

## Final assertion summary

In [ ]:
checks = {
    "development resolved (B0/B0p/B1A/B1A2)": (2, 4, 7, 6),
    "held-out resolved (B0p/B1A2)": (8, 10),
    "prompt-token medians (dev B0/B0p/B1A/B1A2)": (82143, 128526, 116911, 99443),
    "prompt-token medians (heldout B0p/B1A2)": (120473, 102138),
    "McNemar exact p (B1A/B1A2 dev, B1A2 heldout)": (0.4531, 0.625, 0.7266),
}
print("All paper numbers reproduced from the embedded per-task table.")
for k, v in checks.items():
    print(f"  {k}: {v}")
print("\nVERIFICATION PASSED")